In [ ]:
import pandas as pd
import joblib
from pathlib import Path
from IPython.display import display

processed_dir = Path("../data/processed")
models_dir    = Path("../models")

In [6]:
print("=" * 60)
print("🏗️ BUILDING WORLD CUP INFERENCE DATASET (MATCHED PLAYERS)")
print("=" * 60)

# 1. Define paths
mapping_path = Path("../data/processed/squad_to_tm_mapping.csv")
features_path = Path("../data/processed/model_dataset_final_v2.csv")
output_path = Path("../data/processed/world_cup_inference_dataset.csv")

# 2. Load data
df_mapping = pd.read_csv(mapping_path)
df_features = pd.read_csv(features_path)

# Ensure date is datetime for accurate sorting
df_features['date'] = pd.to_datetime(df_features['date'])

# 3. Filter mapping for the 929 successfully matched players
df_matched = df_mapping[df_mapping['transfermarkt_player_id'].notna()].copy()

# 4. Extract latest features from the warehouse
print("Extracting the most recent match data for matched players...")
mapped_ids = df_matched['transfermarkt_player_id'].unique()

# Filter warehouse for matched IDs
df_warehouse = df_features[df_features['player_id'].isin(mapped_ids)].copy()

# Sort chronologically and isolate the absolute latest row per player
df_latest = df_warehouse.sort_values(by=['player_id', 'date'], ascending=[True, False]) \
                        .drop_duplicates(subset=['player_id'], keep='first')

# 5. Merge mapping info with the latest features
print("Merging mapping table with feature snapshot...")
df_inference = pd.merge(
    df_matched[['wc_player_id', 'name', 'transfermarkt_player_id', 'match_type']],
    df_latest,
    left_on='transfermarkt_player_id',
    right_on='player_id',
    how='inner'
)

# 6. Isolate the exact columns expected by XGBoost (+ ID cols for mapping)
model_features = [
    'position', 'market_value_in_eur', 
    'avg_fp_last_3', 'avg_fp_last_5', 'std_fp_last_5', 
    'avg_goals_last_5', 'avg_assists_last_5', 'avg_minutes_last_5', 'matches_played_last_5',
    'prev_season_xG', 'prev_season_xA', 'prev_season_npxG', 'prev_season_key_passes'
]

final_cols = ['wc_player_id', 'name', 'transfermarkt_player_id'] + model_features

# Check for any missing columns and fill with 0 to prevent pipeline breakage
missing_cols = [col for col in final_cols if col not in df_inference.columns]
for col in missing_cols:
    df_inference[col] = 0

df_inference = df_inference[final_cols].copy()

# 7. Audit Output
print("\n--- Inference Dataset Audit ---")
print(f"Target Rows:   {len(df_matched)}")
print(f"Actual Rows:   {len(df_inference)}")
print(f"Total Columns: {len(df_inference.columns)}")

if len(df_inference) < len(df_matched):
    print(f"⚠️ Note: {len(df_matched) - len(df_inference)} matched players had no historical data in the warehouse.")

# 8. Save
df_inference.to_csv(output_path, index=False)
print(f"\n✅ Inference dataset saved to: {output_path.name}")
print("=" * 60)

🏗️ BUILDING WORLD CUP INFERENCE DATASET (MATCHED PLAYERS)
Extracting the most recent match data for matched players...
Merging mapping table with feature snapshot...

--- Inference Dataset Audit ---
Target Rows:   929
Actual Rows:   900
Total Columns: 16
⚠️ Note: 29 matched players had no historical data in the warehouse.

✅ Inference dataset saved to: world_cup_inference_dataset.csv


In [ ]:
print("=" * 60)
print("🔎 STEP 2: DATASET VALIDATION & GAP FIX (FEATURE ALIGNMENT)")
print("=" * 60)

# 1. Load the data
inference_path = Path("../data/processed/world_cup_inference_dataset.csv")
mapping_path = Path("../data/processed/squad_to_tm_mapping.csv")
squad_path = Path("../scrapers/fifa_squads/output/wc2026_players.json") 
model_path = Path("../models/xgb_fantasy_model_v1.joblib")

df_inference = pd.read_csv(inference_path)
df_mapping = pd.read_csv(mapping_path)
df_squads = pd.read_json(squad_path)

if 'wc_player_id' not in df_squads.columns:
    df_squads['wc_player_id'] = df_squads.index

# 2. Re-integrate the 29 missing players
df_matched_only = df_mapping[df_mapping['transfermarkt_player_id'].notna()].copy()
df_full_929 = pd.merge(
    df_matched_only[['wc_player_id', 'name', 'transfermarkt_player_id']], 
    df_inference.drop(columns=['name', 'transfermarkt_player_id'], errors='ignore'), 
    on='wc_player_id', 
    how='left'
)

# ---------------------------------------------------------
# 3. EXACT FEATURE ALIGNMENT (Order and Completeness)
# ---------------------------------------------------------
# This list is explicitly derived from your XGBoost model's expected schema
model_features = [
    'market_value_in_eur', 
    'position', 
    'avg_fp_last_3', 
    'avg_fp_last_5', 
    'avg_goals_last_5', 
    'avg_assists_last_5', 
    'avg_minutes_last_5', 
    'std_fp_last_5', 
    'matches_played_last_5', 
    'prev_season_games', 
    'prev_season_xG', 
    'prev_season_xA', 
    'prev_season_shots', 
    'prev_season_key_passes', 
    'prev_season_npxG', 
    'prev_season_xGChain', 
    'prev_season_xGBuildup'
]

# Check if any expected features are missing entirely from our data tracking and add them
missing_features = [col for col in model_features if col not in df_full_929.columns]
if missing_features:
    print(f"⚠️ Injecting missing schema columns from training: {missing_features}")
    for col in missing_features:
        df_full_929[col] = 0

# Smart Imputation for Position & Numeric columns
pos_dtype = df_inference['position'].dtype if 'position' in df_inference.columns else 'object'
mode_position = df_inference['position'].mode()[0] if 'position' in df_inference.columns else 'M'

df_full_929['position'] = df_full_929['position'].fillna(mode_position).astype(pos_dtype)
df_full_929['market_value_in_eur'] = df_full_929['market_value_in_eur'].fillna(50000)

numeric_cols = [col for col in model_features if col != 'position']
df_full_929[numeric_cols] = df_full_929[numeric_cols].fillna(0)

# Attach 'country' for distribution validation
df_full_929 = pd.merge(df_full_929, df_squads[['wc_player_id', 'country']], on='wc_player_id', how='left')

# --- VALIDATION AUDIT ---
print("\n--- A. Missing Values Check ---")
missing_counts = df_full_929[model_features].isna().sum()
print("✅ No missing values in model features." if missing_counts.sum() == 0 else missing_counts[missing_counts > 0])

print("\n--- B. Feature Completeness ---")
print(f"Rows: {len(df_full_929)} (Expected: 929)")
print(f"Inference Input Shape: {df_full_929[model_features].shape} (Must match model features list length)")

print("\n" + "=" * 60)
print("🤖 STEPS 3 & 4: MODEL LOADING & PREDICTION")
print("=" * 60)

if not model_path.exists():
    print(f"❌ Model not found at {model_path}")
else:
    print(f"Loading champion model: {model_path.name}...")
    xgb_model = joblib.load(model_path)
    
    # Enable categorical support if model requires it and position is parsed as string/object
    if str(pos_dtype) in ['object', 'category'] and type(xgb_model).__name__ == 'XGBRegressor':
        xgb_model.set_params(enable_categorical=True)
        df_full_929['position'] = df_full_929['position'].astype('category')

    # Slice and sort the exact columns in the precise order expected by XGBoost
    X_inference = df_full_929[model_features]
    
    # Execute Predictions
    print("Executing predictions for all 929 players...")
    df_full_929['predicted_fantasy_points'] = xgb_model.predict(X_inference)
    df_full_929['predicted_fantasy_points'] = df_full_929['predicted_fantasy_points'].round(2)
    
    print("\n--- Output Audit: Top 5 Predicted Players ---")
    display_cols = ['name', 'country', 'position', 'predicted_fantasy_points', 'avg_fp_last_5']
    top_preds = df_full_929.sort_values(by='predicted_fantasy_points', ascending=False).head(5)
    display(top_preds[display_cols])
    
    # Save the predicted dataset
    final_out_path = Path("../data/processed/world_cup_predictions_929.csv")
    df_full_929.to_csv(final_out_path, index=False)
    print(f"\n✅ Predictions complete and saved to: {final_out_path.name}")

🔎 STEP 2: DATASET VALIDATION & GAP FIX (FEATURE ALIGNMENT)
⚠️ Injecting missing schema columns from training: ['prev_season_games', 'prev_season_shots', 'prev_season_xGChain', 'prev_season_xGBuildup']

--- A. Missing Values Check ---
✅ No missing values in model features.

--- B. Feature Completeness ---
Rows: 929 (Expected: 929)
Inference Input Shape: (929, 17) (Must match model features list length)

🤖 STEPS 3 & 4: MODEL LOADING & PREDICTION
Loading champion model: xgb_fantasy_model_v1.joblib...
Executing predictions for all 929 players...

--- Output Audit: Top 5 Predicted Players ---


,name,country,position,predicted_fantasy_points,avg_fp_last_5
662,Erling HAALAND,Norway,0,8.81,6.0
914,Darwin NUNEZ,Uruguay,0,7.02,1.0
208,RAPHINHA,Brazil,0,6.88,8.2
451,Kylian MBAPPE,France,0,6.69,4.4
206,VINICIUS JUNIOR,Brazil,0,6.30,5.8



✅ Predictions complete and saved to: world_cup_predictions_929.csv


In [12]:
print("=" * 60)
print("🏗️ REBUILDING WORLD CUP INFERENCE DATASET WITH ALL 17 FEATURES")
print("=" * 60)

# 1. Define paths
mapping_path = Path("../data/processed/squad_to_tm_mapping.csv")
features_path = Path("../data/processed/model_dataset_final_v2.csv")
output_path = Path("../data/processed/world_cup_inference_dataset.csv")

# 2. Load data
df_mapping = pd.read_csv(mapping_path)
df_features = pd.read_csv(features_path)

# Ensure date is datetime for accurate sorting
df_features['date'] = pd.to_datetime(df_features['date'])

# 3. Filter mapping for the 929 successfully matched players
df_matched = df_mapping[df_mapping['transfermarkt_player_id'].notna()].copy()

# 4. Extract latest features from the warehouse
print("Extracting the most recent match data for matched players...")
mapped_ids = df_matched['transfermarkt_player_id'].unique()

# Filter warehouse for matched IDs
df_warehouse = df_features[df_features['player_id'].isin(mapped_ids)].copy()

# Sort chronologically and isolate the absolute latest row per player
df_latest = df_warehouse.sort_values(by=['player_id', 'date'], ascending=[True, False]) \
                        .drop_duplicates(subset=['player_id'], keep='first')

# 5. Merge mapping info with the latest features
print("Merging mapping table with feature snapshot...")
df_inference = pd.merge(
    df_matched[['wc_player_id', 'name', 'transfermarkt_player_id', 'match_type']],
    df_latest,
    left_on='transfermarkt_player_id',
    right_on='player_id',
    how='inner'
)

# 6. UPDATED: Strict 17-feature list expected by XGBoost
model_features = [
    'market_value_in_eur', 
    'position', 
    'avg_fp_last_3', 
    'avg_fp_last_5', 
    'avg_goals_last_5', 
    'avg_assists_last_5', 
    'avg_minutes_last_5', 
    'std_fp_last_5', 
    'matches_played_last_5', 
    'prev_season_games', 
    'prev_season_xG', 
    'prev_season_xA', 
    'prev_season_shots', 
    'prev_season_key_passes', 
    'prev_season_npxG', 
    'prev_season_xGChain', 
    'prev_season_xGBuildup'
]

# Base metadata columns to preserve identities
metadata_cols = ['wc_player_id', 'name', 'transfermarkt_player_id']
final_cols = metadata_cols + model_features

# 7. Check if any Understat columns are completely missing from your warehouse tables
missing_cols = [col for col in final_cols if col not in df_inference.columns]
if missing_cols:
    print(f"⚠️ Note: {missing_cols} missing from warehouse file. Auto-filling with 0.")
    for col in missing_cols:
        df_inference[col] = 0

# Subset the dataframe to contain only these columns
df_inference = df_inference[final_cols].copy()

# 8. Audit Output
print("\n--- Corrected Inference Dataset Audit ---")
print(f"Target Rows:   {len(df_matched)}")
print(f"Actual Rows:   {len(df_inference)}")
print(f"Total Columns: {len(df_inference.columns)} (3 Metadata + 17 Model Features)")

# 9. Overwrite the old CSV file
df_inference.to_csv(output_path, index=False)
print(f"\n✅ Fixed inference dataset saved to: {output_path.name}")
print("=" * 60)

🏗️ REBUILDING WORLD CUP INFERENCE DATASET WITH ALL 17 FEATURES
Extracting the most recent match data for matched players...
Merging mapping table with feature snapshot...

--- Corrected Inference Dataset Audit ---
Target Rows:   929
Actual Rows:   900
Total Columns: 20 (3 Metadata + 17 Model Features)

✅ Fixed inference dataset saved to: world_cup_inference_dataset.csv


In [13]:
print("=" * 60)
print("📊 PRE-FLIGHT VERIFICATION: FEATURE COVERAGE AUDIT (929 PLAYERS)")
print("=" * 60)

# 1. Load data
inference_path = Path("../data/processed/world_cup_inference_dataset.csv")
mapping_path = Path("../data/processed/squad_to_tm_mapping.csv")
squad_path = Path("../scrapers/fifa_squads/output/wc2026_players.json")

df_inference = pd.read_csv(inference_path)
df_mapping = pd.read_csv(mapping_path)
df_squads = pd.read_json(squad_path)

if 'wc_player_id' not in df_squads.columns:
    df_squads['wc_player_id'] = df_squads.index

# 2. Reconstruct the full 929 cohort to check true coverage
df_matched_only = df_mapping[df_mapping['transfermarkt_player_id'].notna()].copy()
df_audit = pd.merge(
    df_matched_only[['wc_player_id', 'name', 'transfermarkt_player_id']], 
    df_inference.drop(columns=['name', 'transfermarkt_player_id'], errors='ignore'), 
    on='wc_player_id', 
    how='left'
)
df_audit = pd.merge(df_audit, df_squads[['wc_player_id', 'country']], on='wc_player_id', how='left')

# 3. Target features to verify
core_features = [
    'position', 
    'market_value_in_eur',
    'avg_fp_last_5', 
    'avg_minutes_last_5', 
    'prev_season_xG', 
    'prev_season_xA',
    'prev_season_games',
    'prev_season_shots',
    'prev_season_xGChain',
    'prev_season_xGBuildup'
]

# 4. Generate Coverage Report
audit_results = []
for col in core_features:
    if col not in df_audit.columns:
        audit_results.append({
            'Feature': col,
            'Status': '❌ MISSING FROM DATAFRAME',
            'Missing Count (NaN)': len(df_audit),
            'Populated/Active Rows': 0,
            'Fill Rate %': 0.0
        })
        continue
        
    nan_count = df_audit[col].isna().sum()
    # Count rows that are populated and not default/zero baseline
    if col == 'position':
        active_count = df_audit[col].notna().sum()
    elif col == 'market_value_in_eur':
        active_count = ((df_audit[col].notna()) & (df_audit[col] > 50000)).sum()
    else:
        active_count = ((df_audit[col].notna()) & (df_audit[col] != 0)).sum()
        
    fill_rate = ((len(df_audit) - nan_count) / len(df_audit)) * 100
    
    audit_results.append({
        'Feature': col,
        'Status': '✅ Ready' if nan_count == 0 else '⚠️ Has Missing Values',
        'Missing Count (NaN)': nan_count,
        'Active/Non-Zero Rows': active_count,
        'Fill Rate %': round(fill_rate, 1)
    })

df_report = pd.DataFrame(audit_results)
print("\n--- Core Feature Matrix Health ---")
display(df_report)

# 5. Cold-Start Identification
cold_start_players = df_audit[df_audit['avg_fp_last_5'].isna() | (df_audit['avg_fp_last_5'] == 0)]
print("\n--- Cohort Breakdown Summary ---")
print(f"Total Matched Players:       {len(df_audit)}")
print(f"Warehouse Active Players:    {len(df_audit) - len(cold_start_players)}")
print(f"Cold-Start Players (0-fill): {len(cold_start_players)} (Will use baseline imputation)")
print("=" * 60)

📊 PRE-FLIGHT VERIFICATION: FEATURE COVERAGE AUDIT (929 PLAYERS)

--- Core Feature Matrix Health ---


,Feature,Status,Missing Count (NaN),Active/Non-Zero Rows,Fill Rate %
0,position,⚠️ Has Missing Values,29,900,96.9
1,market_value_in_eur,⚠️ Has Missing Values,117,808,87.4
2,avg_fp_last_5,⚠️ Has Missing Values,29,893,96.9
3,avg_minutes_last_5,⚠️ Has Missing Values,29,900,96.9
4,prev_season_xG,⚠️ Has Missing Values,29,252,96.9
5,prev_season_xA,⚠️ Has Missing Values,29,239,96.9
6,prev_season_games,⚠️ Has Missing Values,29,321,96.9
7,prev_season_shots,⚠️ Has Missing Values,29,252,96.9
8,prev_season_xGChain,⚠️ Has Missing Values,29,306,96.9
9,prev_season_xGBuildup,⚠️ Has Missing Values,29,297,96.9



--- Cohort Breakdown Summary ---
Total Matched Players:       929
Warehouse Active Players:    893
Cold-Start Players (0-fill): 36 (Will use baseline imputation)


In [14]:
print("=" * 60)
print("🛠️ FINAL PIPELINE: IMPUTATION & WORLD CUP PREDICTION")
print("=" * 60)

# 1. Load Data & Model
inference_path = Path("../data/processed/world_cup_inference_dataset.csv")
model_path = Path("../models/xgb_fantasy_model_v1.joblib")
output_path = Path("../data/processed/world_cup_predictions_final.csv")

df_inference = pd.read_csv(inference_path)
xgb_model = joblib.load(model_path)

# Ensure date features aren't accidentally loaded as text if they exist
if 'date' in df_inference.columns:
    df_inference = df_inference.drop(columns=['date'])

# ---------------------------------------------------------
# 2. THE IMPUTATION PROTOCOL
# ---------------------------------------------------------
print("Applying Baseline Imputation for the 36 Cold-Start Players...")

# A. Handle Positions (Fallback to dataset mode if any are still missing)
pos_dtype = df_inference['position'].dtype
mode_pos = df_inference['position'].mode()[0]
df_inference['position'] = df_inference['position'].fillna(mode_pos).astype(pos_dtype)

# B. Handle Market Value (Minimum baseline of €50k for unknown players)
df_inference['market_value_in_eur'] = df_inference['market_value_in_eur'].fillna(50000)

# C. Handle Form & Understat Metrics (Zero-fill to act as a penalty)
# We isolate just the numeric metrics that require zeroing
model_features = [
    'market_value_in_eur', 'position', 'avg_fp_last_3', 'avg_fp_last_5', 
    'avg_goals_last_5', 'avg_assists_last_5', 'avg_minutes_last_5', 'std_fp_last_5', 
    'matches_played_last_5', 'prev_season_games', 'prev_season_xG', 'prev_season_xA', 
    'prev_season_shots', 'prev_season_key_passes', 'prev_season_npxG', 
    'prev_season_xGChain', 'prev_season_xGBuildup'
]

zero_fill_cols = [col for col in model_features if col not in ['position', 'market_value_in_eur']]
df_inference[zero_fill_cols] = df_inference[zero_fill_cols].fillna(0)

# ---------------------------------------------------------
# 3. VERIFICATION
# ---------------------------------------------------------
missing_count = df_inference[model_features].isna().sum().sum()
if missing_count == 0:
    print("✅ Imputation Complete. Zero missing values remaining in feature matrix.")
else:
    print(f"🚨 Warning: Still have {missing_count} missing values. Check columns.")

# ---------------------------------------------------------
# 4. EXECUTE PREDICTIONS
# ---------------------------------------------------------
print("\nExecuting XGBoost Predictions for all 929 World Cup Players...")

# Enable categorical support if model requires it and position is parsed as string/object
if str(pos_dtype) in ['object', 'category'] and type(xgb_model).__name__ == 'XGBRegressor':
    xgb_model.set_params(enable_categorical=True)
    df_inference['position'] = df_inference['position'].astype('category')

# Slice the exact columns in the precise order expected by XGBoost
X_inference = df_inference[model_features]

# Predict
df_inference['predicted_fantasy_points'] = xgb_model.predict(X_inference)
df_inference['predicted_fantasy_points'] = df_inference['predicted_fantasy_points'].round(2)

# ---------------------------------------------------------
# 5. AUDIT & SAVE
# ---------------------------------------------------------
print("\n--- Output Audit: Top 10 World Cup Projected Players ---")
display_cols = ['name', 'position', 'market_value_in_eur', 'predicted_fantasy_points']

# Clean up market value for display
display_df = df_inference.sort_values(by='predicted_fantasy_points', ascending=False).head(10).copy()
display_df['market_value_in_eur'] = display_df['market_value_in_eur'].map(lambda x: f"€{x:,.0f}")
display(display_df[display_cols])

# Save the final predicted dataset
df_inference.to_csv(output_path, index=False)
print(f"\n✅ Pipeline Complete! Output saved to: {output_path.name}")
print("=" * 60)

🛠️ FINAL PIPELINE: IMPUTATION & WORLD CUP PREDICTION
Applying Baseline Imputation for the 36 Cold-Start Players...
✅ Imputation Complete. Zero missing values remaining in feature matrix.

Executing XGBoost Predictions for all 929 World Cup Players...

--- Output Audit: Top 10 World Cup Projected Players ---


,name,position,market_value_in_eur,predicted_fantasy_points
643,Erling HAALAND,0,"€200,000,000",8.31
442,Kylian MBAPPE,0,"€180,000,000",6.69
205,RAPHINHA,0,"€50,000,000",6.45
885,Darwin NUNEZ,0,"€70,000,000",6.02
203,VINICIUS JUNIOR,0,"€180,000,000",5.75
361,Patrik Schick,0,"€25,000,000",5.00
90,Julian ALVAREZ,0,"€100,000,000",4.93
684,CRISTIANO RONALDO,0,"€15,000,000",4.79
44,Ricardo PEPI,0,"€15,000,000",4.73
641,Jens Petter HAUGE,0,"€2,500,000",4.59



✅ Pipeline Complete! Output saved to: world_cup_predictions_final.csv


In [20]:
print("=" * 60)
print("🤖 RUNNING XGBOOST PREDICTIONS & GENERATING RECOMMENDATIONS")
print("=" * 60)

# 1. Define Paths
inference_path = Path("../data/processed/world_cup_inference_dataset.csv")
squad_path = Path("../scrapers/fifa_squads/output/wc2026_players.json") 
model_path = Path("../models/xgb_fantasy_model_v1.joblib")

# 2. Load Datasets
df_inference = pd.read_csv(inference_path)
df_squads = pd.read_json(squad_path)
xgb_model = joblib.load(model_path)

if 'wc_player_id' not in df_squads.columns:
    df_squads['wc_player_id'] = df_squads.index

# ---------------------------------------------------------
# 1. Attach 'country' from the raw squad file
# ---------------------------------------------------------
if 'country' not in df_inference.columns:
    df_inference = pd.merge(df_inference, df_squads[['wc_player_id', 'country']], on='wc_player_id', how='left')

# ---------------------------------------------------------
# 2. Dynamic Baseline Imputation (Ensures 0 missing values)
# ---------------------------------------------------------
pos_dtype = df_inference['position'].dtype if 'position' in df_inference.columns else 'object'
mode_pos = df_inference['position'].mode()[0] if 'position' in df_inference.columns else 'M'
df_inference['position'] = df_inference['position'].fillna(mode_pos).astype(pos_dtype)

# Minimum baseline market value for cold-start players
df_inference['market_value_in_eur'] = df_inference['market_value_in_eur'].fillna(50000)

# 17 Strict features in exact order expected by XGBoost model
model_features = [
    'market_value_in_eur', 'position', 'avg_fp_last_3', 'avg_fp_last_5', 
    'avg_goals_last_5', 'avg_assists_last_5', 'avg_minutes_last_5', 'std_fp_last_5', 
    'matches_played_last_5', 'prev_season_games', 'prev_season_xG', 'prev_season_xA', 
    'prev_season_shots', 'prev_season_key_passes', 'prev_season_npxG', 
    'prev_season_xGChain', 'prev_season_xGBuildup'
]

# Zero-fill any missing numeric stats
zero_fill_cols = [col for col in model_features if col not in ['position', 'market_value_in_eur']]
df_inference[zero_fill_cols] = df_inference[zero_fill_cols].fillna(0)

# ---------------------------------------------------------
# 3. EXECUTE XGBOOST MODEL PREDICTIONS
# ---------------------------------------------------------
print("Executing model projections for all 929 players...")

if str(pos_dtype) in ['object', 'category'] and type(xgb_model).__name__ == 'XGBRegressor':
    xgb_model.set_params(enable_categorical=True)
    df_inference['position'] = df_inference['position'].astype('category')

# Isolate exact features matrix
X_inference = df_inference[model_features]

# Predict and clip negative values to 0
df_inference['predicted_fantasy_points'] = xgb_model.predict(X_inference)
df_inference['predicted_fantasy_points'] = df_inference['predicted_fantasy_points'].clip(lower=0).round(2)

# ---------------------------------------------------------
# 4. GENERATE RECOMMENDATION ENGINE OUTPUTS
# ---------------------------------------------------------
print("\n🥇 TOP 10 PROJECTED PLAYERS (Overall)")
print("-" * 60)
top_overall = df_inference.sort_values(by='predicted_fantasy_points', ascending=False).head(10)
display(top_overall[['name', 'country', 'position', 'predicted_fantasy_points']])

# Calculate Captaincy Scores (with tiny epsilon to avoid ZeroDivisionError)
df_inference['safe_score'] = df_inference['predicted_fantasy_points'] / (1 + df_inference['std_fp_last_5'] + 0.01)
df_inference['diff_score'] = df_inference['predicted_fantasy_points'] * (1 + df_inference['std_fp_last_5'])

print("\n🛡️ BEST SAFE CAPTAINS (High Floor, Consistent)")
print("-" * 60)
safe_caps = df_inference.sort_values(by='safe_score', ascending=False).head(5)
display(safe_caps[['name', 'country', 'predicted_fantasy_points', 'std_fp_last_5', 'safe_score']])

print("\n🚀 BEST DIFFERENTIAL CAPTAINS (High Volatility, High Ceiling)")
print("-" * 60)
diff_caps = df_inference.sort_values(by='diff_score', ascending=False).head(5)
display(diff_caps[['name', 'country', 'predicted_fantasy_points', 'std_fp_last_5', 'diff_score']])

# Calculate Value Percentiles for Hidden Gems
df_inference['pred_pct'] = df_inference['predicted_fantasy_points'].rank(pct=True) * 100
df_inference['val_pct'] = df_inference['market_value_in_eur'].rank(pct=True) * 100
df_inference['gem_score'] = df_inference['pred_pct'] - df_inference['val_pct']

print("\n💎 TOP 10 HIDDEN GEMS (High Points / Low Market Value)")
print("-" * 60)
# Filter: Must score in top 40% of dataset to be a useful pick
df_playable = df_inference[df_inference['pred_pct'] >= 60].copy()
gems = df_playable.sort_values(by='gem_score', ascending=False).head(10)
gems_display = gems[['name', 'country', 'market_value_in_eur', 'predicted_fantasy_points', 'gem_score']].copy()
gems_display['market_value_in_eur'] = gems_display['market_value_in_eur'].map(lambda x: f"€{x:,.0f}")
display(gems_display)

# ---------------------------------------------------------
# D. TEAM-LEVEL ANALYSIS (Best Attacks)
# ---------------------------------------------------------
print("\n🔥 MOST DANGEROUS ATTACKS (Top 5 Nations)")
print("Based on the combined projections of their top 3 forwards.")
print("-" * 60)


valid_attacker_labels = ['0', '0.0', 'attacker', 'forward', 'a', 'f']
attackers = df_inference[df_inference['position'].astype(str).str.lower().isin(valid_attacker_labels)]

# Sum the top 3 attackers per country
best_attacks = attackers.sort_values(by=['country', 'predicted_fantasy_points'], ascending=[True, False]) \
                        .groupby('country').head(3) \
                        .groupby('country')['predicted_fantasy_points'].sum() \
                        .reset_index() \
                        .sort_values(by='predicted_fantasy_points', ascending=False) \
                        .head(5)

best_attacks.rename(columns={'predicted_fantasy_points': 'Top 3 Attackers Combined Expected Points'}, inplace=True)
display(best_attacks)

# Save final annotated predictions file
final_out_path = Path("../data/processed/world_cup_predictions_final.csv")
df_inference.to_csv(final_out_path, index=False)
print(f"\n✅ Pipeline Complete! All outputs calculated and saved to: {final_out_path.name}")
print("=" * 60)

🤖 RUNNING XGBOOST PREDICTIONS & GENERATING RECOMMENDATIONS
Executing model projections for all 929 players...

🥇 TOP 10 PROJECTED PLAYERS (Overall)
------------------------------------------------------------


,name,country,position,predicted_fantasy_points
643,Erling HAALAND,Norway,0,8.31
442,Kylian MBAPPE,France,0,6.69
205,RAPHINHA,Brazil,0,6.45
885,Darwin NUNEZ,Uruguay,0,6.02
203,VINICIUS JUNIOR,Brazil,0,5.75
361,Patrik Schick,Czechia,0,5.00
90,Julian ALVAREZ,Argentina,0,4.93
684,CRISTIANO RONALDO,Portugal,0,4.79
44,Ricardo PEPI,USA,0,4.73
641,Jens Petter HAUGE,Norway,0,4.59



🛡️ BEST SAFE CAPTAINS (High Floor, Consistent)
------------------------------------------------------------


,name,country,predicted_fantasy_points,std_fp_last_5,safe_score
885,Darwin NUNEZ,Uruguay,6.02,0.0,5.960396
749,Iliman NDIAYE,Senegal,3.40,0.0,3.366337
155,Hans VANAKEN,Belgium,2.45,0.0,2.425743
634,Patrick BERG,Norway,2.39,0.0,2.366337
92,Thiago ALMADA,Argentina,2.37,0.0,2.346535



🚀 BEST DIFFERENTIAL CAPTAINS (High Volatility, High Ceiling)
------------------------------------------------------------


...


💎 TOP 10 HIDDEN GEMS (High Points / Low Market Value)
------------------------------------------------------------


,name,country,market_value_in_eur,predicted_fantasy_points,gem_score
626,Ben WAINE,New Zealand,"€50,000",3.82,89.777778
555,Keisuke GOTO,Japan,"€50,000",3.46,83.388889
862,Deniz GUL,Türkiye,"€50,000",3.43,82.833333
210,RAYAN,Brazil,"€50,000",3.34,81.166667
206,ENDRICK,Brazil,"€50,000",3.32,80.277778
238,Richard RIOS,Colombia,"€50,000",3.12,74.000000
869,Fernando MUSLERA,Uruguay,"€750,000",3.59,73.333333
375,Anthony VALENCIA,Ecuador,"€1,000,000",3.67,71.666667
136,Michael Gregoritsch,Austria,"€1,500,000",3.82,69.777778
525,MERCHAS DOSKI,Iraq,"€50,000",3.04,69.611111



🔥 MOST DANGEROUS ATTACKS (Top 5 Nations)
Based on the combined projections of their top 3 forwards.
------------------------------------------------------------


,country,Top 3 Attackers Combined Expected Points
31,Norway,17.050001
6,Brazil,16.059999
17,France,14.809999
1,Argentina,13.660000
13,Côte d'Ivoire,12.540000



✅ Pipeline Complete! All outputs calculated and saved to: world_cup_predictions_final.csv
